In [ ]:
# Cell 1 — preflight, locate source, install the physics dependency, and prepare outputs.
from pathlib import Path
from datetime import datetime, timezone
import importlib.util
import json
import shutil
import subprocess
import sys
import uuid

import numpy as np
import pandas as pd

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

# Require the successful Day 6 checkpoint, not the older 5K baseline.
valid_runs = []
for summary_path in INPUT.rglob("day6_run_summary.json"):
    try:
        candidate = json.loads(summary_path.read_text(encoding="utf-8"))
        inventory = candidate.get("inventory", {})
        stages = candidate.get("stages", [])
        stages_ok = bool(stages) and all(item.get("status") == "complete" for item in stages)
        if (
            candidate.get("target_scored_reached") is True
            and int(inventory.get("completed_or_archived_products", 0)) >= 20_000
            and int(inventory.get("feature_observations", 0)) >= 20_000
            and int(candidate.get("isolation_forest_scored_rows", 0)) >= 20_000
            and stages_ok
        ):
            valid_runs.append((summary_path, candidate))
    except Exception as exc:
        print(f"Skipping unreadable run summary {summary_path}: {exc}")

if not valid_runs:
    raise FileNotFoundError(
        "Attach the successful Day 6 20K notebook output containing day6_run_summary.json. "
        "The older 5K baseline is not enough for this preflight."
    )
DAY6_SUMMARY_PATH, DAY6_SUMMARY = valid_runs[0]
print("[PASS] Successful 20K baseline:", DAY6_SUMMARY_PATH)
print("      Observations:", DAY6_SUMMARY["inventory"]["feature_observations"])
print("      Unique TICs:", DAY6_SUMMARY["inventory"]["unique_tic_sources_with_features"])
print("      Isolation Forest scored:", DAY6_SUMMARY["isolation_forest_scored_rows"])

project_candidates = []
for pyproject in INPUT.rglob("pyproject.toml"):
    root = pyproject.parent
    if all((root / relative).exists() for relative in (
        "src/cosmic_black_swan/simulate.py",
        "src/cosmic_black_swan/inference.py",
        "src/cosmic_black_swan/flows.py",
    )):
        project_candidates.append(root)
if not project_candidates:
    raise FileNotFoundError(
        "Attach Cosmic Swan 4 (or another input containing cbs-project-src with simulate.py, inference.py, and flows.py)."
    )
PROJECT_SOURCE = project_candidates[0]
print("[PASS] Physics source package:", PROJECT_SOURCE)

free_gib = shutil.disk_usage(WORK).free / (1024 ** 3)
if free_gib < 5.0:
    raise RuntimeError(f"Only {free_gib:.2f} GiB is free; this run needs at least 5 GiB.")
print(f"[PASS] Working space: {free_gib:.2f} GiB free")

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:6]
RUN_ROOT = WORK / f"day7-physics-{RUN_ID}"
PROJECT = RUN_ROOT / "cbs-project-src"
RUN_ROOT.mkdir(parents=True, exist_ok=False)
shutil.copytree(PROJECT_SOURCE, PROJECT)

# Register the copied src tree in this live kernel before installing.
# This avoids relying on Kaggle's editable-install path handling across notebook cells.
PACKAGE_SRC = PROJECT / "src"
PACKAGE_DIR = PACKAGE_SRC / "cosmic_black_swan"
if not (PACKAGE_DIR / "simulate.py").is_file():
    raise FileNotFoundError(f"Copied source package is incomplete: {PACKAGE_DIR}")
if str(PACKAGE_SRC) not in sys.path:
    sys.path.insert(0, str(PACKAGE_SRC))

# Install the project for its declared dependencies and CLI; direct sys.path registration
# above makes local package imports deterministic in the current Kaggle kernel.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT)], check=True)
from cosmic_black_swan.simulate import simulate_prior_predictive  # noqa: E402
print("[PASS] Package import:", Path(simulate_prior_predictive.__code__.co_filename).resolve())
if importlib.util.find_spec("numpyro") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpyro>=0.15"], check=True)

import torch
if not torch.cuda.is_available():
    raise RuntimeError("CUDA PyTorch is not visible. In Kaggle Settings, select GPU T4 x2 before running this notebook.")
print("[PASS] PyTorch GPU:", torch.cuda.get_device_name(0))

import jax
import numpyro
print("NumPyro:", numpyro.__version__)
print("JAX backend/devices:", jax.default_backend(), jax.devices())
print("Simulation output root:", RUN_ROOT)

STAGE_LOG = []
STAGE_LOG_PATH = RUN_ROOT / "day7_stage_log.json"

def save_stage_log():
    STAGE_LOG_PATH.write_text(json.dumps(STAGE_LOG, indent=2, default=str), encoding="utf-8")

def run_stage(name, function):
    started = datetime.now(timezone.utc)
    print(f"\n===== {name} =====", flush=True)
    try:
        result = function()
        row = {"stage": name, "status": "complete", "result": result}
        print(json.dumps(row, indent=2, default=str), flush=True)
    except Exception as exc:
        row = {"stage": name, "status": "failed", "error": f"{type(exc).__name__}: {exc}"}
        print(row["error"], flush=True)
    row["seconds"] = round((datetime.now(timezone.utc) - started).total_seconds(), 1)
    STAGE_LOG.append(row)
    save_stage_log()
    return row

print("PREFLIGHT PASSED. No simulations have started in this cell.")

In [ ]:
# Cell 2 — generate 100,000 unique simulations for each physical family.
# The package import is verified in Cell 1; this import reuses the registered source path.
from cosmic_black_swan.simulate import simulate_prior_predictive

DRAWS_PER_FAMILY = 100_000
SEQUENCE_LENGTH = 1297  # approximately 30-minute sampling over a 27-day binary baseline
SIMULATION_SEEDS = {"binary": 701, "white_dwarf": 702, "zeeman": 703}

for family in ("binary", "white_dwarf", "zeeman"):
    def simulate_one(family=family):
        result = simulate_prior_predictive(
            output_dir=RUN_ROOT / "simulations",
            family=family,
            parameter_draws=DRAWS_PER_FAMILY,
            noise_realizations=1,
            seed=SIMULATION_SEEDS[family],
            chunk_size=512,
            sequence_length=SEQUENCE_LENGTH,
        )
        metadata = json.loads((RUN_ROOT / "simulations" / family / "simulation_metadata.json").read_text())
        if metadata["parameter_draws_unique"] != DRAWS_PER_FAMILY:
            raise RuntimeError(f"{family}: simulator did not produce the requested unique parameter draws")
        result["sampling_diagnostics"] = metadata.get("sampling_diagnostics", metadata.get("spectral_resolution_diagnostics"))
        print(f"Free disk after {family}: {shutil.disk_usage(WORK).free / (1024 ** 3):.2f} GiB")
        return result
    run_stage(f"prior-predictive simulation: {family}", simulate_one)

print("Requested total unique parameter draws:", 3 * DRAWS_PER_FAMILY)

In [ ]:
# Cell 3 — train held-out conditional RealNVP flows for the supported families.
# The package deliberately does not train a flow for its one-parameter white-dwarf inversion.
from cosmic_black_swan.flows import train_conditional_flow

FLOW_EPOCHS = 20
FLOW_BATCH_SIZE = 1024

for family in ("binary", "zeeman"):
    simulation_dir = RUN_ROOT / "simulations" / family
    metadata_file = simulation_dir / "simulation_metadata.json"
    if not metadata_file.exists():
        row = {"stage": f"conditional RealNVP: {family}", "status": "skipped", "reason": "simulation stage did not produce metadata"}
        STAGE_LOG.append(row)
        save_stage_log()
        print(row)
        continue

    def fit_flow(simulation_dir=simulation_dir):
        result = train_conditional_flow(
            simulation_dir=simulation_dir,
            epochs=FLOW_EPOCHS,
            batch_size=FLOW_BATCH_SIZE,
            learning_rate=1e-3,
            seed=42,
            eval_sources=1000,
            posterior_samples=256,
        )
        print("Flow device:", "cuda" if torch.cuda.is_available() else "cpu")
        return result
    run_stage(f"conditional RealNVP held-out coverage: {family}", fit_flow)

In [ ]:
# Cell 4 — make known-truth synthetic observations and run NumPyro NUTS recovery tests.
from cosmic_black_swan.physics import binary_eclipse_model, white_dwarf_radius_solar, zeeman_triplet_spectrum
from cosmic_black_swan.inference import run_numpyro_nuts

VALIDATION_DIR = RUN_ROOT / "nuts_validation_inputs"
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(704)
truth = {}

# Binary: inject a known eclipsing binary into a 27-day, ~30-minute-cadence series.
t_binary = np.linspace(0.0, 27.0, SEQUENCE_LENGTH)
binary_truth = {
    "period_days": 2.4,
    "phase": 0.24,
    "depth_primary": 0.16,
    "depth_ratio": 0.45,
    "duration_fraction": 0.08,
    "ellipsoidal_amplitude": 0.004,
    "reflection_amplitude": -0.002,
    "extra_noise": 0.0015,
}
binary_flux = binary_eclipse_model(
    t_binary,
    binary_truth["period_days"], binary_truth["phase"],
    binary_truth["depth_primary"], binary_truth["depth_ratio"],
    binary_truth["duration_fraction"], binary_truth["ellipsoidal_amplitude"],
    binary_truth["reflection_amplitude"],
)
binary_flux = binary_flux + rng.normal(0.0, binary_truth["extra_noise"], len(t_binary))
binary_error = np.full(len(t_binary), 0.0008)
pd.DataFrame({"time_days": t_binary, "flux": binary_flux, "flux_err": binary_error}).to_csv(VALIDATION_DIR / "binary.csv", index=False)
truth["binary"] = binary_truth

# White dwarf: one noisy radius measurement from the zero-temperature mass-radius curve.
wd_mass = 0.80
wd_radius_true = float(white_dwarf_radius_solar(wd_mass))
wd_radius_error = wd_radius_true * 0.04
wd_radius_observed = wd_radius_true + rng.normal(0.0, wd_radius_error)
pd.DataFrame({"radius_solar": [wd_radius_observed], "radius_error_solar": [wd_radius_error]}).to_csv(VALIDATION_DIR / "white_dwarf.csv", index=False)
truth["white_dwarf"] = {"mass_solar": wd_mass, "radius_true_solar": wd_radius_true, "radius_observed_solar": wd_radius_observed}

# Zeeman: toy H-alpha triplet with a known field, velocity, depth, and width.
wavelength = np.linspace(6500.0, 6625.0, SEQUENCE_LENGTH)
zeeman_truth = {
    "line_center_angstrom": 6562.8,
    "field_gauss": 18_000.0,
    "radial_velocity_km_s": 35.0,
    "line_depth": 0.60,
    "line_sigma_angstrom": 0.35,
    "intrinsic_noise": 0.002,
}
zeeman_flux = zeeman_triplet_spectrum(
    wavelength,
    line_center_angstrom=zeeman_truth["line_center_angstrom"],
    magnetic_field_gauss=zeeman_truth["field_gauss"],
    radial_velocity_km_s=zeeman_truth["radial_velocity_km_s"],
    line_depth=zeeman_truth["line_depth"],
    line_sigma_angstrom=zeeman_truth["line_sigma_angstrom"],
)
zeeman_flux += rng.normal(0.0, zeeman_truth["intrinsic_noise"], len(wavelength))
zeeman_error = np.full(len(wavelength), 0.0015)
pd.DataFrame({"wavelength_angstrom": wavelength, "normalized_flux": zeeman_flux, "flux_err": zeeman_error}).to_csv(VALIDATION_DIR / "zeeman.csv", index=False)
truth["zeeman"] = zeeman_truth
(VALIDATION_DIR / "known_truth.json").write_text(json.dumps(truth, indent=2), encoding="utf-8")
print("Saved three known-truth validation inputs:", VALIDATION_DIR)

for index, family in enumerate(("binary", "white_dwarf", "zeeman")):
    def fit_nuts(family=family, index=index):
        return run_numpyro_nuts(
            family=family,
            input_path=VALIDATION_DIR / f"{family}.csv",
            output_dir=RUN_ROOT / "physics_posteriors",
            warmup=500,
            samples=1000,
            chains=4,
            seed=710 + index,
            field_max_gauss=20_000.0,
        )
    run_stage(f"NumPyro NUTS synthetic recovery: {family}", fit_nuts)

In [ ]:
# Cell 5 — summarize simulation counts, held-out flow coverage, NUTS diagnostics, and artifacts.
from pathlib import Path

simulation_summary = {}
flow_summary = {}
nuts_summary = {}
for family in ("binary", "white_dwarf", "zeeman"):
    metadata_path = RUN_ROOT / "simulations" / family / "simulation_metadata.json"
    if metadata_path.exists():
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        simulation_summary[family] = {
            "unique_parameter_draws": metadata.get("parameter_draws_unique"),
            "noise_realizations_per_draw": metadata.get("noise_realizations_per_draw"),
            "simulated_observations": metadata.get("simulated_observation_count"),
            "observation_shape": metadata.get("observation_shape"),
            "sampling_diagnostics": metadata.get("sampling_diagnostics", metadata.get("spectral_resolution_diagnostics")),
        }
    else:
        simulation_summary[family] = {"status": "missing; simulation stage failed"}

    flow_path = RUN_ROOT / "simulations" / family / "flow" / "flow_run.json"
    if flow_path.exists():
        flow_run = json.loads(flow_path.read_text(encoding="utf-8"))
        flow_summary[family] = {
            "family": flow_run.get("family"),
            "training_records": flow_run.get("training_records"),
            "heldout_records": flow_run.get("heldout_records"),
            "eval_parameter_draws": flow_run.get("eval_parameter_draws"),
            "heldout_interval_coverage": flow_run.get("heldout_interval_coverage"),
            "interpretation": flow_run.get("interpretation"),
        }

    nuts_path = RUN_ROOT / "physics_posteriors" / family / "nuts_run.json"
    if nuts_path.exists():
        nuts_run = json.loads(nuts_path.read_text(encoding="utf-8"))
        nuts_summary[family] = {
            "sampling_diagnostics": nuts_run.get("sampling_diagnostics"),
            "posterior_summary": nuts_run.get("posterior_summary"),
            "model_assumptions": nuts_run.get("model_assumptions"),
        }

successful_draws = sum(
    int(item.get("unique_parameter_draws", 0))
    for item in simulation_summary.values() if isinstance(item, dict)
)
report = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "day6_baseline_summary": str(DAY6_SUMMARY_PATH),
    "simulation_draws_total": successful_draws,
    "simulations_by_family": simulation_summary,
    "normalizing_flow_results": flow_summary,
    "numpyro_nuts_results": nuts_summary,
    "stage_log": STAGE_LOG,
    "artifact_root": str(RUN_ROOT),
    "claim_limits": [
        "These are synthetic-prior experiments, not physical inference on real TESS candidates.",
        "The eclipsing-binary generator is a trapezoidal approximation; check cadence and period aliases.",
        "The white-dwarf inference assumes the zero-temperature Nauenberg mass-radius relation and one radius measurement.",
        "The Zeeman spectrum is a weak-field toy triplet and omits the instrumental line-spread function and line identification.",
        "Flow coverage is evaluated on held-out simulations from the same declared simulator priors; it does not validate real-star posterior coverage.",
        "A code run and simulation count alone do not establish model adequacy; review coverage, posterior predictive fit, R-hat, ESS, and divergences.",
    ],
}
summary_path = WORK / "day7_physics_summary.json"
summary_path.write_text(json.dumps(report, indent=2, default=str), encoding="utf-8")
print("Total unique simulated parameter draws:", successful_draws)
print("Run summary:", summary_path)
print(json.dumps({
    "simulation_counts": {k: v.get("unique_parameter_draws") for k, v in simulation_summary.items()},
    "flow_families_completed": sorted(flow_summary),
    "nuts_families_completed": sorted(nuts_summary),
    "stage_status": [{"stage": x["stage"], "status": x["status"], "seconds": x.get("seconds")} for x in STAGE_LOG],
    "artifact_root": str(RUN_ROOT),
}, indent=2, default=str))